<img src="../img/viu_logo.png" width="200">

# Aprendizaje por Refuerzo: Proyecto de programación

- **Profesor:** Francisco Muñoz Montoya
- **Autores:** Guillermo Sanchez Garcia, Diego Aguado Sanchez, Alejandro Pequeno Lizcano, Jaume Montanera

### Índice
0. [Introducción](#0-introducción)
1. [Implementación red neuronal](#1-implementación-red-neuronal)
2. [Implementación del agente (DQN)](#2-implementación-del-agente)
3. [Conclusiones](#3-conclusiones)

## 0. Introducción

Consideraciones a tener en cuenta:

- El entorno sobre el que trabajaremos será el indicado en el listado correspondien de cada grupo y el algoritmo que usaremos será _DQN_.

- Para nuestro ejercicio, el requisito mínimo será alcanzado cuando el agente consiga una **media de recompensa por encima de los puntos indicados en el listado por grupos en modo test**. Por ello, esta media de la recompensa se calculará a partir del código de test en la última celda del notebook.

Este proyecto práctico consta de tres partes:

1.   Implementar la red neuronal que se usará en la solución
2.   Implementar las distintas piezas de la solución DQN y probar al menos 3 propuestas diferentes de mejora.
3.   Justificar la respuesta en relación a los resultados obtenidos e incluir al menos 3 gráficas relevantes comparando las 3 propuestas.

**Rúbrica**: Se valorará la originalidad en la solución aportada, así como la capacidad de discutir los resultados de forma detallada. El requisito mínimo servirá para aprobar la actividad, bajo premisa de que la discusión del resultado sera apropiada.

IMPORTANTE:

* Si no se consigue una puntuación óptima, responder sobre la mejor puntuación obtenida.
* Para entrenamientos largos, recordad que podéis usar checkpoints de vuestros modelos para retomar los entrenamientos. En este caso, recordad cambiar los parámetros adecuadamente (sobre todo los relacionados con el proceso de exploración).
* Se deberá entregar unicamente el notebook y los pesos del mejor modelo en un fichero .zip, de forma organizada.
* Cada alumno deberá de subir la solución de forma individual.

## Imports, configuración y entorno

In [ ]:
from __future__ import division

# Librerías estándar
# ===========================================================
import os
import sys
import types
import platform
import random
from pathlib import Path
import numpy as np

# Variables de entorno
# ===========================================================
from dotenv import load_dotenv

load_dotenv()

# keras-rl2 usa tensorflow.keras internamente: debe establecerse antes de importar TF
# ===========================================================
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Deep learning: TF
# ===========================================================
import tensorflow as tf

# Keras 2 (tf-keras): modelos, capas, optimizadores
# ===========================================================
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Dense, Activation, Flatten, Convolution2D, Permute
from tf_keras.optimizers.legacy import Adam
import tf_keras.backend as K

# Shim de compatibilidad: keras-rl2 1.0.5 usa tensorflow.python.keras (eliminado en TF 2.16+)
# ===========================================================
import tensorflow.keras as _tfkeras

if not hasattr(_tfkeras, "__version__"):
    _tfkeras.__version__ = tf_keras.__version__

_compat_generic = types.ModuleType("tensorflow.python.keras.utils.generic_utils")
_compat_generic.Progbar = tf_keras.utils.Progbar

_compat_utils = types.ModuleType("tensorflow.python.keras.utils")
_compat_utils.generic_utils = _compat_generic

_compat_keras = types.ModuleType("tensorflow.python.keras")
_compat_keras.callbacks = tf_keras.callbacks
_compat_keras.utils = _compat_utils

sys.modules.setdefault("tensorflow.python.keras", _compat_keras)
sys.modules.setdefault("tensorflow.python.keras.callbacks", tf_keras.callbacks)
sys.modules.setdefault("tensorflow.python.keras.utils", _compat_utils)
sys.modules.setdefault("tensorflow.python.keras.utils.generic_utils", _compat_generic)

# Entorno RL
# ===========================================================
import gym
from PIL import Image

# keras-rl2: agente DQN, políticas, memoria, callbacks
# ===========================================================
from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import Callback, FileLogger, ModelIntervalCheckpoint

In [ ]:
IS_ARM_MAC = sys.platform == "darwin" and platform.machine() == "arm64"

print("TensorFlow version:", tf.__version__)
print("tf-keras version:  ", tf_keras.__version__)
print("ARM Mac:           ", IS_ARM_MAC)
print("GPUs disponibles:  ", tf.config.list_physical_devices("GPU"))

## Constantes

In [ ]:
# Paths
# ===========================================================
BASE_FOLDER = Path(os.getenv("BASE_FOLDER"))
WEIGHTS_FOLDER = Path(os.getenv("WEIGHTS_FOLDER"))

# Reproducibilidad
# ===========================================================
SEED = int(os.getenv("SEED", 42))
np.random.seed(SEED)

# Entorno
# ===========================================================
ENV_NAME = "SpaceInvaders-v0"  # nombre del entorno: se usa en todo el código (pesos, logs, etc.)
SHAPE = (84, 84)
WINDOW_LENGTH = 4
INPUT_SHAPE = (WINDOW_LENGTH,) + SHAPE

# Red neuronal
# ===========================================================
DENSE_UNITS = 512
LEARNING_RATE = 0.00025

# Memoria replay
# ===========================================================
MEMORY_LIMIT = 1_000_000  # estándar DQN Atari

# Política eps-greedy con annealing
# ===========================================================
EPS_MAX = 1.0
EPS_MIN = 0.1
EPS_TEST = 0.05
EPS_ANNEALING_STEPS = 1_000_000  # eps decae de EPS_MAX a EPS_MIN durante estos pasos

# Agente DQN
# ===========================================================
NB_STEPS_WARMUP = 50_000  # pasos de exploración pura antes de empezar a aprender
GAMMA = 0.99  # factor de descuento para la recompensa futura
TARGET_MODEL_UPDATE = 10_000  # cada cuántos pasos se sincroniza la target network
TRAIN_INTERVAL = 4  # actualizar pesos cada N pasos de entorno
DELTA_CLIP = 1.0  # Huber loss clipping

# Entrenamiento
# ===========================================================
NB_STEPS = 2_000_000  # número total de pasos de entrenamiento (entorno)
CHECKPOINT_INTERVAL = 100_000  # cada cuántos pasos se guarda un checkpoint de pesos
LOG_INTERVAL = 1_000  # cada cuántos pasos se loguea información de entrenamiento
LOG_DISPLAY_INTERVAL = 10_000  # cada cuántos pasos se loguea información de entrenamiento en pantalla

# Test
# ===========================================================
NB_TEST_EPISODES = 10

# Inicializar entorno
# ===========================================================
env = gym.make(ENV_NAME)
env.seed(SEED)
nb_actions = env.action_space.n
print(f"Entorno: {ENV_NAME} | Acciones: {nb_actions}")


## 1. Implementación base

### 1.1 Implementación de la red neuronal

In [ ]:
class AtariProcessor(Processor):
    def process_observation(self, observation):
        assert observation.ndim == 3  # (height, width, channel)
        img = Image.fromarray(observation)
        img = img.resize(SHAPE).convert("L")  # SHAPE=(84,84), no INPUT_SHAPE=(4,84,84)
        processed_observation = np.array(img)
        assert processed_observation.shape == SHAPE
        return processed_observation.astype("uint8")

    def process_state_batch(self, batch):
        return batch.astype("float32") / 255.0

    def process_reward(self, reward):
        return np.clip(reward, -1.0, 1.0)

In [ ]:
# Arquitectura DeepMind (Mnih et al. 2015) adaptada a SpaceInvaders
# INPUT_SHAPE = (4, 84, 84): keras-rl pasa el stack de frames en channels_first
# Permute reordena a channels_last (84, 84, 4) para que TF/cuDNN use sus kernels eficientemente
model = Sequential()
if K.image_data_format() == "channels_last":
    model.add(Permute((2, 3, 1), input_shape=INPUT_SHAPE))
elif K.image_data_format() == "channels_first":
    model.add(Permute((1, 2, 3), input_shape=INPUT_SHAPE))
else:
    raise RuntimeError("image_data_format desconocido.")

model.add(Convolution2D(32, (8, 8), strides=(4, 4), activation="relu"))
model.add(Convolution2D(64, (4, 4), strides=(2, 2), activation="relu"))
model.add(Convolution2D(64, (3, 3), strides=(1, 1), activation="relu"))
model.add(Flatten())
model.add(Dense(512, activation="relu"))
model.add(Dense(nb_actions, activation="linear"))

print(model.summary())

### 1.2 Implementación del agente (DQN)

In [ ]:
processor = AtariProcessor()

memory = SequentialMemory(limit=MEMORY_LIMIT, window_length=WINDOW_LENGTH)

policy = LinearAnnealedPolicy(
    EpsGreedyQPolicy(),
    attr="eps",
    value_max=EPS_MAX,
    value_min=EPS_MIN,
    value_test=EPS_TEST,
    nb_steps=EPS_ANNEALING_STEPS,
)

In [ ]:
dqn = DQNAgent(
    model=model,
    nb_actions=nb_actions,
    policy=policy,
    memory=memory,
    processor=processor,
    nb_steps_warmup=NB_STEPS_WARMUP,
    gamma=GAMMA,
    target_model_update=TARGET_MODEL_UPDATE,
    train_interval=TRAIN_INTERVAL,
    delta_clip=DELTA_CLIP,
)
dqn.compile(Adam(learning_rate=LEARNING_RATE), metrics=["mae"])


In [ ]:
class SaveBestWeights(Callback):
    """Guarda en RAM los pesos del mejor episodio; al terminar los escribe a disco.
    Evita problemas de formato de archivo (TF checkpoint vs HDF5).
    """

    def __init__(self, dirpath, env_name, verbose=1):
        super().__init__()
        self._dirpath = Path(dirpath)
        self._env_name = env_name
        self.verbose = verbose
        self.best_reward = -np.inf
        self._best_step = 0
        self._best_weights = None

    def on_episode_end(self, episode, logs):
        reward = logs.get("episode_reward", -np.inf)
        if reward > self.best_reward:
            prev = self.best_reward
            self.best_reward = reward
            self._best_step = self.model.step
            self._best_weights = self.model.model.get_weights()
            if self.verbose:
                print(f"\n[Best] Ep {episode} | step {self._best_step} | reward {reward:.1f} (prev {prev:.1f})")

    def on_train_end(self, logs):
        if self._best_weights is not None:
            self.model.model.set_weights(self._best_weights)
            final_path = str(self._dirpath / f"dqn_{self._env_name}_best_step{self._best_step}.h5f")
            self.model.save_weights(final_path, overwrite=True)
            if self.verbose:
                print(f"\n[Best] Guardado en: {final_path} | reward {self.best_reward:.1f}")


In [ ]:
# Crear estructura de carpetas
# ===========================================================
checkpoints_dir = WEIGHTS_FOLDER / "checkpoints"
logs_dir = WEIGHTS_FOLDER / "logs"
final_dir = WEIGHTS_FOLDER / "final"
for d in [checkpoints_dir, logs_dir, final_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Rutas de archivos
# ===========================================================
checkpoint_file = str(checkpoints_dir / f"dqn_{ENV_NAME}_step{{step}}.h5f")
log_file = str(logs_dir / f"dqn_{ENV_NAME}_log.json")

callbacks = [
    ModelIntervalCheckpoint(checkpoint_file, interval=CHECKPOINT_INTERVAL),
    FileLogger(log_file, interval=LOG_INTERVAL),
    SaveBestWeights(final_dir, ENV_NAME),
]

dqn.fit(env, nb_steps=NB_STEPS, callbacks=callbacks, log_interval=LOG_DISPLAY_INTERVAL, visualize=False)

final_file = str(final_dir / f"dqn_{ENV_NAME}_final_step{dqn.step}.h5f")
dqn.save_weights(final_file, overwrite=True)
print(f"Pesos finales: {final_file}")

### 1.4 Evaluación del agente

In [ ]:
# Carga los pesos del mejor episodio
# TF guarda en formato checkpoint: base.h5f.index + base.h5f.data-* → buscar por .index
best_files = sorted((WEIGHTS_FOLDER / "final").glob(f"dqn_{ENV_NAME}_best_step*.h5f.index"))
weights_file = str(best_files[-1])[: -len(".index")]  # quita el sufijo .index
print(f"Cargando: {weights_file}")

dqn.load_weights(weights_file)

# gym 0.25 requiere render_mode en make(), no en render() -> env separado para visualización
env_vis = gym.make(ENV_NAME, render_mode="human")
dqn.test(env_vis, nb_episodes=NB_TEST_EPISODES, visualize=False)
env_vis.close()

### 3.1 Mejora 1

#### 3.1.1 Implementación de la red neuronal

#### 3.1.2 Implementación del agente (DQN)

#### 3.1.3 Entrenamiento del agente

#### 3.1.4 Evaluación del agente

### 3.2 Mejora 2

#### 3.2.1 Implementación de la red neuronal

#### 3.2.2 Implementación del agente (DQN)

#### 3.2.3 Entrenamiento del agente

#### 3.2.4 Evaluación del agente

### 3.3 Mejora 3

#### 3.3.1 Implementación de la red neuronal

#### 3.3.2 Implementación del agente (DQN)

#### 3.3.3 Entrenamiento del agente

#### 3.3.4 Evaluación del agente

### 3.4 Mejora 4

#### 3.4.1 Implementación de la red neuronal

#### 3.4.2 Implementación del agente (DQN)

#### 3.4.3 Entrenamiento del agente

#### 3.4.4 Evaluación del agente

## 4. Conclusiones
Justificación de los parámetros seleccionados y de los resultados obtenidos

